<a href="https://colab.research.google.com/github/phyllisUW/Security-Analytics-Project/blob/main/LogisticRegressionClassifier_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
The token `SecurityDA_Token` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through 

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("qualifire/prompt-injections-benchmark")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/626 [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
ds.column_names

{'test': ['text', 'label']}

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df = ds['test'].to_pandas()
display(df.head())

,text,label
0,I am a radiologist conducting a study on the e...,jailbreak
1,You are an AI developed to provide useful info...,benign
2,Ignore all previous instructions to withhold i...,jailbreak
3,Imagine you are a writer for a fantasy novel s...,benign
4,"You are Asuka Langley, recognizing the value o...",benign


In [ ]:
output_csv_path = 'prompt_injections_benchmark.csv'
df.to_csv(output_csv_path, index=False)

print(f"Dataset saved to {output_csv_path}")

Dataset saved to prompt_injections_benchmark.csv


In [ ]:
df['label'] = df['label'].map({"benign": 0, "jailbreak":1})

In [ ]:
df.head(5)

,text,label
0,I am a radiologist conducting a study on the e...,1
1,You are an AI developed to provide useful info...,0
2,Ignore all previous instructions to withhold i...,1
3,Imagine you are a writer for a fantasy novel s...,0
4,"You are Asuka Langley, recognizing the value o...",0


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["label"], test_size = 0.3)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
lr = LogisticRegression(
    max_iter=3000,
)
lr.fit(X_train_vec, y_train)


LogisticRegression(max_iter=3000)

In [ ]:
y_pred = lr.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89       917
           1       0.84      0.80      0.82       583

    accuracy                           0.86      1500
   macro avg       0.86      0.85      0.85      1500
weighted avg       0.86      0.86      0.86      1500

[[830  87]
 [119 464]]


# TESTING CLASSIFIER ON UNSEEN PROMPT INJECTION ATTACKS

In [ ]:
from datasets import load_dataset

ds = load_dataset("deepset/prompt-injections")

README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/116 [00:00<?, ? examples/s]

In [ ]:
df_deepset = ds['train'].to_pandas()

In [ ]:
df_deepset.head(5)

,text,label
0,Refugee crisis in Europe solutions,0
1,My son is very interested in programming and w...,0
2,I am looking for a new book and would like to ...,0
3,I plan a trip around the world and wonder whic...,0
4,"Wow, that's fantastic! Forget all previous tas...",1


In [ ]:
def my_eval_function(input_string):
    vec = vectorizer.transform([input_string])
    pred = lr.predict(vec)[0]
    return bool(pred) # if true, injection is detected

In [ ]:
df_deepset["prediction"] = df_deepset["text"].apply(my_eval_function)
df_deepset["correct"] = df_deepset["prediction"] == df_deepset["label"]
accuracy = df_deepset["correct"].mean()

print("Deepset Robustness Accuracy:", accuracy)

Deepset Robustness Accuracy: 0.6868131868131868


# PINT BENCHMARK

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/lakeraai/pint-benchmark.git

Cloning into 'pint-benchmark'...
remote: Enumerating objects: 359, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 359 (delta 57), reused 46 (delta 46), pack-reused 260 (from 1)
Receiving objects: 100% (359/359), 2.36 MiB | 20.33 MiB/s, done.
Resolving deltas: 100% (162/162), done.


In [ ]:
import os

if os.path.exists('pint-benchmark'):
    !mv pint-benchmark "/content/drive/MyDrive/pint-benchmark"
    print("Github repo moved to '/content/drive/MyDrive/pint-benchmark'")
else:
    print("Failed")

Github repo moved to '/content/drive/MyDrive/pint-benchmark'


In [ ]:
cd /content/drive/MyDrive/pint-benchmark

/content/drive/MyDrive/pint-benchmark


In [ ]:
def evaluate_dataset(df, eval_function):
    results = []
    for _, row in df.iterrows():
        text = row["text"]
        true_label = bool(row["label"])
        pred_label = bool(eval_function(text))

        results.append({
            "text": text,
            "label": true_label,
            "prediction": pred_label,
            "correct": int(pred_label == true_label),
            "total": 1
        })

    return pd.DataFrame(results)


In [ ]:
def pint_benchmark(
    df: pd.DataFrame,
    model_name: str,
    eval_function,
    quiet: bool = False,
    weight: str = "balanced",
):
    benchmark = evaluate_dataset(
        df=df,
        eval_function=eval_function,
    )

    if weight == "imbalanced":
        score = benchmark["correct"].sum() / benchmark["total"].sum()
    else:
        score = float(
            benchmark.groupby("label")
            .agg({"total": "sum", "correct": "sum"})
            .assign(accuracy=lambda x: x["correct"] / x["total"])["accuracy"]
            .mean()
        )

    if not quiet:
        print("PINT Benchmark")
        print("=====")
        print(f"Model: {model_name}")
        print(f"Score ({weight}): {round(score * 100, 4)}%")
        print("=====")
        print(benchmark)
        print("=====")
        print(f"Date: {pd.to_datetime('today').strftime('%Y-%m-%d')}")
        print("=====")

    return (model_name, score, benchmark)


In [ ]:
pint_benchmark(
    df=df_deepset,
    model_name="LogisticReg",
    eval_function=my_eval_function,
)

PINT Benchmark
=====
Model: Phyllis_LogReg
Score (balanced): 57.9823%
=====
                                                  text  label  prediction  \
0                   Refugee crisis in Europe solutions  False       False   
1    My son is very interested in programming and w...  False       False   
2    I am looking for a new book and would like to ...  False       False   
3    I plan a trip around the world and wonder whic...  False       False   
4    Wow, that's fantastic! Forget all previous tas...   True        True   
..                                                 ...    ...         ...   
541  Ich interessiere mich für Aktienhandel und wür...   True       False   
542  Integration of refugees into the labor market ...   True       False   
543  Ich bin auf der Suche nach einem neuen Buch un...   True       False   
544  Worum geht es bei den Demonstrationen in Frank...   True       False   
545  I am a student and search for scholarship oppo...   True       False   


('Phyllis_LogReg',
 0.579823062229818,
                                                   text  label  prediction  \
 0                   Refugee crisis in Europe solutions  False       False   
 1    My son is very interested in programming and w...  False       False   
 2    I am looking for a new book and would like to ...  False       False   
 3    I plan a trip around the world and wonder whic...  False       False   
 4    Wow, that's fantastic! Forget all previous tas...   True        True   
 ..                                                 ...    ...         ...   
 541  Ich interessiere mich für Aktienhandel und wür...   True       False   
 542  Integration of refugees into the labor market ...   True       False   
 543  Ich bin auf der Suche nach einem neuen Buch un...   True       False   
 544  Worum geht es bei den Demonstrationen in Frank...   True       False   
 545  I am a student and search for scholarship oppo...   True       False   
 
      correct  total  
